# 01.02 Anthropic Claude

`ChatAnthropic`은 Claude 계열(Opus·Sonnet·Haiku) 호출의 표준 진입점이다. 툴콜·스트리밍은 물론 **extended thinking**(확장 사고) 파라미터로 reasoning 예산을 명시할 수 있다.

## 학습 목표

- `ChatAnthropic(model=..., max_tokens=...)` 기본 사용법을 익힌다
- `thinking={"type": "enabled", "budget_tokens": ...}`로 확장 사고를 제어한다
- `bind_tools([...])`와 `stream()` 을 조합해 도구 호출 스트리밍을 관찰한다
- Anthropic 고유의 `usage_metadata.input_token_details`(cache_creation/cache_read) 를 읽는다

## 언제 쓰나

- 장문 reasoning이 필요한 에이전트(리서치 · 코딩 · 분석)
- Constitutional AI / Responsible Scaling Policy 요건에 맞는 고위험 워크로드
- 1시간 prompt cache + 긴 system prompt 워크로드 (→ `11_provider_middleware/01_anthropic_prompt_caching.ipynb`)

## 01.02.1 환경 설정

`langchain-anthropic` 과 `.env`의 `ANTHROPIC_API_KEY` 가 필요하다.

In [ ]:
# !pip install -U langchain langchain-anthropic
import os
from dotenv import load_dotenv
load_dotenv(override=True)

assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY 필요"

# live probe — 키가 실제로 유효한지 확인(잘못된/만료된 키도 여기서 실패)
from langchain_anthropic import ChatAnthropic
ChatAnthropic(model="claude-haiku-4-5", max_tokens=8).invoke("ping")

## 01.02.2 기본 사용

`max_tokens`는 **출력** 한계다. Anthropic은 다른 공급자와 달리 `max_tokens`가 필수 인자임에 주의한다.

In [ ]:
from langchain_anthropic import ChatAnthropic

model = ChatAnthropic(model="claude-sonnet-4-6", max_tokens=1024)
msg = model.invoke("LangChain에서 ChatAnthropic의 가장 큰 차별점을 한 문장으로 설명해줘.")
print(msg.content)
print("usage:", msg.usage_metadata)

## 01.02.3 `init_chat_model()` 경로

`init_chat_model("anthropic:claude-sonnet-4-6")`로도 동일하게 만든다.

In [ ]:
from langchain.chat_models import init_chat_model

m = init_chat_model("anthropic:claude-sonnet-4-6", max_tokens=512)
print(m.invoke("핑").content)

## 01.02.4 Extended thinking — reasoning budget

`thinking={"type": "enabled", "budget_tokens": N}` 를 주면 모델은 응답 전 N 토큰을 **thinking block**에 쓴다. `max_tokens`는 `budget_tokens + 최종 답변` 합보다 커야 한다.

In [ ]:
thinker = ChatAnthropic(
    model="claude-sonnet-4-6",
    max_tokens=4096,
    thinking={"type": "enabled", "budget_tokens": 2048},
)
r = thinker.invoke("3자리 수 747을 소인수분해한 뒤, 그 과정을 설명해줘.")
# content는 thinking block + text block 리스트
for block in r.content:
    kind = block.get("type") if isinstance(block, dict) else type(block).__name__
    print("block:", kind)

## 01.02.5 Tool calling + streaming

`bind_tools(...)`는 OpenAI와 같은 인터페이스. 스트리밍 시 tool 호출은 `tool_call_chunks`로 분할 수신된다.

In [ ]:
from langchain_core.tools import tool

@tool
def weather(city: str) -> str:
    """도시의 현재 날씨를 반환한다."""
    return f"{city}: 맑음 24C"

bound = model.bind_tools([weather])

chunks = []
for chunk in bound.stream("서울 날씨 알려줘."):
    chunks.append(chunk)
final = chunks[0]
for c in chunks[1:]:
    final = final + c
print("tool_calls:", final.tool_calls)

## 01.02.6 Structured output

Anthropic은 내부적으로 tool-based 구조화 출력 경로를 사용한다.

In [ ]:
from pydantic import BaseModel

class Weather(BaseModel):
    city: str
    condition: str
    temp_c: float

structured = model.with_structured_output(Weather)
print(structured.invoke("부산은 흐리고 22도야. 구조화해줘."))

## 01.02.7 `.profile` 로 능력치 확인

In [ ]:
p = model.profile
print("max_input_tokens :", p.get("max_input_tokens"))
print("tool_calling    :", p.get("tool_calling"))
print("reasoning_output :", p.get("reasoning_output"))
print("structured_output:", p.get("structured_output"))

## 체크리스트

- [ ] `max_tokens` 를 반드시 지정했다 (Anthropic 필수)
- [ ] `thinking` 예산과 `max_tokens`의 관계(예산 ≤ max_tokens)를 이해했다
- [ ] `usage_metadata.input_token_details`에서 cache_creation/cache_read 가 보인다
- [ ] `bind_tools` + `stream()` 조합으로 tool_call_chunks 를 관찰했다

## 다음

- `11_provider_middleware/01_anthropic_prompt_caching.ipynb` — Prompt caching middleware
- `11_provider_middleware/02_claude_bash_tool.ipynb` — Claude 내장 bash tool

## 참고

- 공식: https://docs.langchain.com/oss/python/integrations/chat/anthropic
- Extended thinking: https://docs.anthropic.com/en/docs/build-with-claude/extended-thinking